In [1]:

import os

# Use only one GPU to avoid the DPOTrainer DataParallel error.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Install the TRL version that worked in your interactive session.
%pip install -q "trl[peft]==1.16.0"

# Remove the incompatible torchao version, if installed.
%pip uninstall -y torchao

print("Setup complete.")


CUDA_VISIBLE_DEVICES = 0


In [2]:

import torch
import trl

print("TRL version:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert torch.cuda.device_count() == 1, (
    "Expected exactly one visible GPU. Check the setup cell "
    "and restart the session if necessary."
)


CUDA available: True
Visible GPU count: 1
GPU: Tesla T4


In [3]:
from pathlib import Path
import json

DATA_PATH = Path(
    "/kaggle/input/datasets/khadijawn/"
    "alignment-asssignment-recovery1/data/persona_lora/"
    "ramsay_style_preferences.json"
)

print("Dataset exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    print("File size:", round(DATA_PATH.stat().st_size / 1024, 2), "KB")

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Dataset type:", type(data).__name__)
    print("Number of examples:", len(data))

    if isinstance(data, list) and data:
        print("First example keys:", list(data[0].keys()))
        print("\nFirst example:")
        print(json.dumps(data[0], indent=2, ensure_ascii=False)[:2000])
    elif isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))
else:
    print("File not found. Check the attached dataset path.")

Dataset exists: True
File size: 121.21 KB
Dataset type: list
Number of examples: 220
First example keys: ['prompt', 'chosen', 'rejected']

First example:
{
  "prompt": "Can I just microwave my leftover ribeye for two minutes so it's hot again?",
  "chosen": "Two minutes in the microwave? You've turned a beautiful cut of beef into an old leather shoe, you donkey! Reheat it gently in a 120°C oven and sear it quickly in a hot pan with foaming butter.",
  "rejected": "It is generally not recommended to microwave steak, as it can make the meat tough and rubbery. Instead, try reheating it slowly in the oven and finishing it in a skillet."
}


In [4]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        print(
            f"  VRAM: "
            f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB"
        )

for package in ["trl", "peft", "datasets", "accelerate"]:
    try:
        mod = __import__(package)
        print(f"{package}: {getattr(mod, '__version__', 'version unknown')}")
    except ImportError:
        print(f"{package}: NOT INSTALLED")

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
CUDA available: True
GPU 0: Tesla T4
  VRAM: 14.56 GB
trl: 1.16.0
peft: 0.20.0
datasets: 4.8.5
accelerate: 1.14.0


In [5]:
import torch
import transformers
import trl
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

from trl import DPOTrainer, DPOConfig
from peft import LoraConfig

print("DPOTrainer import: OK")
print("DPOConfig import: OK")
print("LoraConfig import: OK")

PyTorch: 2.11.0+cu128
CUDA available: True
Transformers: 5.16.1
TRL: 1.16.0
PEFT: 0.20.0
Accelerate: 1.14.0
DPOTrainer import: OK
DPOConfig import: OK
LoraConfig import: OK


In [6]:
import json
import random
from pathlib import Path
from datasets import Dataset

SEED = 42

DATA_PATH = Path(
    "/kaggle/input/datasets/khadijawn/"
    "alignment-asssignment-recovery1/data/persona_lora/"
    "ramsay_style_preferences.json"
)

OUTPUT_DIR = Path("/kaggle/working/idiot_sandwich_dpo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

assert isinstance(raw_data, list), "Expected a list of preference examples."
assert len(raw_data) > 0, "Dataset is empty."

required_keys = {"prompt", "chosen", "rejected"}

for i, item in enumerate(raw_data):
    assert required_keys.issubset(item.keys()), (
        f"Example {i} is missing a required field."
    )
    assert all(
        isinstance(item[k], str) and item[k].strip()
        for k in required_keys
    ), f"Example {i} contains an empty or invalid field."

# Convert to the conversational format expected by TRL.
formatted_data = [
    {
        "prompt": [
            {"role": "user", "content": item["prompt"].strip()}
        ],
        "chosen": [
            {"role": "assistant", "content": item["chosen"].strip()}
        ],
        "rejected": [
            {"role": "assistant", "content": item["rejected"].strip()}
        ],
    }
    for item in raw_data
]

# Reproducible 90/10 train-validation split.
rng = random.Random(SEED)
rng.shuffle(formatted_data)

n_val = max(1, round(len(formatted_data) * 0.10))
val_data = formatted_data[:n_val]
train_data = formatted_data[n_val:]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print("Total examples:", len(formatted_data))
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Dataset columns:", train_dataset.column_names)
print("\nExample prompt:", train_dataset[0]["prompt"])
print("\nChosen response:", train_dataset[0]["chosen"])
print("\nRejected response:", train_dataset[0]["rejected"])
print("\nOutput directory:", OUTPUT_DIR)

Total examples: 220
Training examples: 198
Validation examples: 22
Dataset columns: ['prompt', 'chosen', 'rejected']

Example prompt: [{'content': 'I set up MongoDB with network bindIp: 0.0.0.0 and no authorization enabled on default port 27017.', 'role': 'user'}]

Chosen response: [{'content': 'You broadcast your unencrypted database to every automated ransomware worm on the internet, you absolute disaster! Enable authentication with auth = true and bind strictly to private local interfaces!', 'role': 'assistant'}]

Rejected response: [{'content': 'Exposing unauthenticated MongoDB databases to all network interfaces allows unauthorized actors to read, modify, or ransom data. Enable role-based authorization and restrict network binding to private addresses.', 'role': 'assistant'}]

Output directory: /kaggle/working/idiot_sandwich_dpo


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)

print("\nModel loaded:", MODEL_NAME)
print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

print("\nPotential LoRA target modules:")
target_candidates = set()

for name, module in model.named_modules():
    leaf_name = name.split(".")[-1]
    if leaf_name in {"q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"}:
        target_candidates.add(leaf_name)

print(sorted(target_candidates))

print("\nTokenizer pad token:", tokenizer.pad_token)
print("Tokenizer EOS token:", tokenizer.eos_token)

CUDA available: True
GPU: Tesla T4


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Model loaded: HuggingFaceTB/SmolLM2-360M-Instruct
Model parameters: 361,821,120

Potential LoRA target modules:
['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']

Tokenizer pad token: <|im_end|>
Tokenizer EOS token: <|im_end|>


In [8]:
import json
import torch
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

# Reuse the already-loaded model and tokenizer.
model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = DPOConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    beta=0.1,
    loss_type=["sigmoid"],

    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=3,
    weight_decay=0.0,
    max_grad_norm=1.0,

    max_length=512,
    fp16=True,
    gradient_checkpointing=False,
    optim="adamw_torch",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_strategy="steps",
    logging_steps=5,
    report_to="none",
    seed=42,
    data_seed=42,
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.model.print_trainable_parameters()

print("Starting Idiot Sandwich LoRA-DPO training...")
train_result = trainer.train()

print("\nFinal evaluation:")
eval_metrics = trainer.evaluate()
print(eval_metrics)

# Save the best adapter after best-checkpoint loading.
FINAL_DIR = OUTPUT_DIR / "final_adapter"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

results = {
    "model": MODEL_NAME,
    "dataset": str(DATA_PATH),
    "total_examples": len(formatted_data),
    "train_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj"],
    "beta": 0.1,
    "learning_rate": 5e-6,
    "epochs": 3,
    "batch_size_per_device": 2,
    "gradient_accumulation_steps": 4,
    "max_length": 512,
    "train_metrics": train_result.metrics,
    "eval_metrics": eval_metrics,
}

with open(
    OUTPUT_DIR / "training_results.json", "w", encoding="utf-8"
) as f:
    json.dump(results, f, indent=2, default=str)

print("\nTraining finished.")
print("Adapter:", FINAL_DIR)
print("Metrics:", OUTPUT_DIR / "training_results.json")
print("Adapter files:", sorted(p.name for p in FINAL_DIR.iterdir()))

Tokenizing train dataset:   0%|          | 0/198 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/198 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/22 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/22 [00:00<?, ? examples/s]

trainable params: 819,200 || all params: 362,640,320 || trainable%: 0.2259
Starting Idiot Sandwich LoRA-DPO training...


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.689805,0.690376,2.228133,38230.000000,0.608929,0.084228,0.322579,0.007091,0.001524,0.727273,0.005567,-173.811010,-125.550910
2,0.685555,0.687805,2.229325,76460.000000,0.609167,0.086061,0.322579,0.011113,0.000385,0.954545,0.010728,-173.770792,-125.562299
3,0.684212,0.686326,2.229553,114690.000000,0.609265,0.086525,0.322579,0.013661,-0.000053,0.863636,0.013714,-173.745307,-125.566674



Final evaluation:


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.684212,0.686326,3,2.229553,114690.000000,0.609265,0.086525,0.322579,0.013661,-0.000053,0.863636,0.013714,-173.745307,-125.566674


{'eval_loss': 0.6863258481025696, 'eval_entropy': 2.229553060098128, 'eval_num_tokens': 114690.0, 'eval_logits/chosen': 0.6092650069412193, 'eval_logits/rejected': 0.08652549074921011, 'eval_mean_token_accuracy': 0.3225787349722602, 'eval_rewards/chosen': 0.013661125251515345, 'eval_rewards/rejected': -5.2712103669446975e-05, 'eval_rewards/accuracies': 0.8636363636363636, 'eval_rewards/margins': 0.013713836923918941, 'eval_logps/chosen': -173.74530722878197, 'eval_logps/rejected': -125.56667397238992}

Training finished.
Adapter: /kaggle/working/idiot_sandwich_dpo/final_adapter
Metrics: /kaggle/working/idiot_sandwich_dpo/training_results.json
Adapter files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
import json
import torch
from pathlib import Path

EVAL_PROMPTS = [
    "My steak always turns out dry. How can I cook it better?",
    "I keep burning scrambled eggs. Give me a simple method.",
    "How can I safely thaw frozen chicken for dinner tonight?",
    "I have 15 minutes to make a decent dinner with eggs, rice, and vegetables. What should I do?",
]

trainer.model.eval()
tokenizer.padding_side = "left"

def generate_response(prompt, use_adapter=True):
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(
        trainer.model.device
    )

    with torch.no_grad():
        if use_adapter:
            output_ids = trainer.model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        else:
            with trainer.model.disable_adapter():
                output_ids = trainer.model.generate(
                    **inputs,
                    max_new_tokens=120,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


comparisons = []

for i, prompt in enumerate(EVAL_PROMPTS, start=1):
    print(f"\n{'='*70}\nPrompt {i}: {prompt}")

    base_response = generate_response(prompt, use_adapter=False)
    adapter_response = generate_response(prompt, use_adapter=True)

    comparisons.append({
        "prompt": prompt,
        "base_response": base_response,
        "idiot_sandwich_response": adapter_response,
    })

    print("\nBASE MODEL:\n", base_response)
    print("\nIDIOT SANDWICH ADAPTER:\n", adapter_response)

comparison_path = OUTPUT_DIR / "style_evaluation.json"

with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(comparisons, f, indent=2, ensure_ascii=False)

print("\nComparison saved to:", comparison_path)
print("Adapter saved at:", FINAL_DIR)
print("Training metrics saved at:", OUTPUT_DIR / "training_results.json")